In [ ]:
# Importing libraries
import numpy as np
import pandas as pd
import joblib


# Loading the pre-trained data preprocessor and ensemble model package
pre_processor=joblib.load('complete_preprocessor_RFS.pkl')
ensemble_pkg=joblib.load('ensemble_package_RFS.pkl')

data_path = '/content/FinalTestDataset2025.xls'
test_df = pd.read_excel(data_path)

# Function to preprocess test data using the loaded preprocessor
def preprocessinf(df,prep):
  df = df.copy()
  df = df.replace(999, np.nan)
  df['Gene_Flag'] = df['Gene'].isna().astype(int) # Creating a flag for missing 'Gene' values (1 if missing, 0 if present)
  df[prep['clinical_mode'].index] = df[prep['clinical_mode'].index].fillna(prep['clinical_mode'])
  df[prep['radiomic_features']] = df[prep['radiomic_features']].fillna(prep['radiomic_median'])

  if len(prep['skewed_features']) > 0:  # Applying log transformation to skewed features, if there are any present
        df[prep['skewed_features']] = np.log1p(df[prep['skewed_features']]) 

  df[prep['radiomic_features']] = prep['scaler'].transform(df[prep['radiomic_features']]) # Standardize radiomic features using the scaler from the preprocessor

  return df[prep['combined_features']]


X_test=preprocessinf(test_df,pre_processor)
predictions = np.column_stack([
    model.predict(X_test) for model in ensemble_pkg['models'].values()
])
final_RFS = predictions.mean(axis=1)
# Create a DataFrame 
RFS_df = pd.DataFrame({
    'PatientID': test_df['ID'].astype(str),
    'RelapseFreeSurvival (outcome)': final_RFS.round(3)
})

RFS_df.to_csv('RFSPrediction.csv', index=False, float_format="%.3f")


